# Tensor Parallel Advanced: Attention and MLP Parallelism

## Overview

Deep dive into tensor parallelism for Transformer attention and MLP layers.

### Topics Covered
- Attention head parallelism
- MLP column/row parallel
- Communication optimization
- Memory analysis

## 1. Attention Parallelism

### Multi-Head Attention Split

```
Original: 32 heads on 1 GPU
TP=8: 4 heads per GPU

GPU 0: heads 0-3   → Q0,K0,V0 → Attn0 → O0
GPU 1: heads 4-7   → Q1,K1,V1 → Attn1 → O1
...                                      ↓
GPU 7: heads 28-31 → Q7,K7,V7 → Attn7 → O7
                                         ↓
                              AllReduce(O0+O1+...+O7)
```

In [ ]:
import torch
import torch.nn as nn

class TensorParallelAttention(nn.Module):
    """Multi-head attention with tensor parallelism."""
    
    def __init__(self, hidden_size, num_heads, tp_size, tp_rank):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.tp_size = tp_size
        self.tp_rank = tp_rank
        
        # Each GPU handles num_heads/tp_size heads
        self.heads_per_gpu = num_heads // tp_size
        self.head_dim = hidden_size // num_heads
        self.local_hidden = self.heads_per_gpu * self.head_dim
        
        # QKV projection (column parallel)
        self.qkv = nn.Linear(hidden_size, 3 * self.local_hidden, bias=False)
        
        # Output projection (row parallel)
        self.out_proj = nn.Linear(self.local_hidden, hidden_size, bias=False)
    
    def forward(self, x):
        B, S, H = x.shape
        
        # QKV projection
        qkv = self.qkv(x)  # [B, S, 3*local_hidden]
        q, k, v = qkv.chunk(3, dim=-1)
        
        # Reshape for attention
        q = q.view(B, S, self.heads_per_gpu, self.head_dim).transpose(1, 2)
        k = k.view(B, S, self.heads_per_gpu, self.head_dim).transpose(1, 2)
        v = v.view(B, S, self.heads_per_gpu, self.head_dim).transpose(1, 2)
        
        # Attention
        attn = torch.matmul(q, k.transpose(-2, -1)) / (self.head_dim ** 0.5)
        attn = torch.softmax(attn, dim=-1)
        out = torch.matmul(attn, v)
        
        # Reshape and project
        out = out.transpose(1, 2).contiguous().view(B, S, self.local_hidden)
        out = self.out_proj(out)  # Needs AllReduce after this
        
        return out

# Example
attn = TensorParallelAttention(hidden_size=4096, num_heads=32, tp_size=8, tp_rank=0)
print(f"Heads per GPU: {attn.heads_per_gpu}")
print(f"Local hidden: {attn.local_hidden}")

## 2. MLP Parallelism

```
MLP: X → Linear1 → GeLU → Linear2 → Y

Tensor Parallel:
X → [Column Parallel] → GeLU → [Row Parallel] → AllReduce → Y
     (no comm)                  (partial sums)
```

In [ ]:
class TensorParallelMLP(nn.Module):
    """MLP with tensor parallelism."""
    
    def __init__(self, hidden_size, intermediate_size, tp_size, tp_rank):
        super().__init__()
        self.tp_size = tp_size
        
        # Column parallel: split output dimension
        local_intermediate = intermediate_size // tp_size
        self.fc1 = nn.Linear(hidden_size, local_intermediate, bias=False)
        
        # Row parallel: split input dimension
        self.fc2 = nn.Linear(local_intermediate, hidden_size, bias=False)
        
        self.activation = nn.GELU()
    
    def forward(self, x):
        # Column parallel (no communication needed)
        x = self.fc1(x)
        x = self.activation(x)
        
        # Row parallel (needs AllReduce after)
        x = self.fc2(x)
        
        # AllReduce would happen here in distributed setting
        return x

mlp = TensorParallelMLP(4096, 16384, tp_size=8, tp_rank=0)
print(f"FC1: {mlp.fc1.in_features} -> {mlp.fc1.out_features}")
print(f"FC2: {mlp.fc2.in_features} -> {mlp.fc2.out_features}")

## 3. Communication Analysis

| Layer | Forward | Backward | Total per Layer |
|-------|---------|----------|----------------|
| Attention | 1 AllReduce | 1 AllReduce | 2 AllReduce |
| MLP | 1 AllReduce | 1 AllReduce | 2 AllReduce |
| **Total** | 2 AllReduce | 2 AllReduce | **4 AllReduce** |

## 4. Summary

### Key Points

1. **Attention**: Split heads across GPUs
2. **MLP**: Column parallel → Row parallel pattern
3. **Communication**: 4 AllReduce per transformer layer
4. **Best for**: Large hidden dimensions, within-node parallelism